# AETHER — Stage 6: Planner Thought -> Speaker Soft-Prompt Probe

Переосмысление после Stage 5 (см. `docs/reports/technical_report_03.md`): вместо hidden state
из уже сгенерированного текста Speaker'а (постфактум, циклично), Planner теперь пишет расширенную
внутреннюю "мысль" (структуру ответа, не финальные слова), и её hidden state прокидывается в
Speaker как soft-prompt — виртуальный токен, вставленный перед обычным промптом через
`inputs_embeds`.

**Это структурный probe, не тест качества.** Мост (`ThoughtBridge`) необучен (случайная
инициализация) — проверяем только: работает ли сам канал (детерминированность, влияние на
генерацию, специфичность к конкретной мысли), а не "стал ли ответ лучше". Обучение моста и
оценка качества — Stage 7/8, не этот прогон.

In [ ]:
REPO_URL = "https://github.com/Manifestro/aether.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
MODEL_ID = "Qwen/Qwen3-1.7B"  # @param {type:"string"}

if "YOUR_USERNAME" in REPO_URL:
    raise ValueError("Укажи настоящий REPO_URL")


In [ ]:
import os, subprocess, sys
from pathlib import Path

subprocess.run(["nvidia-smi"], check=False)
repo_dir = Path("/content/aether")
if (repo_dir / ".git").exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{repo_dir}[dev,ml]"], check=True)
print("Commit:")
subprocess.run(["git", "rev-parse", "HEAD"], check=True)


In [ ]:
artifacts = repo_dir / "artifacts" / "colab-stage6"
artifacts.mkdir(parents=True, exist_ok=True)
tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
(artifacts / "tests.log").write_text(tests.stdout, encoding="utf-8")
print(tests.stdout)
if tests.returncode != 0:
    raise RuntimeError("Tests failed")


In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(repo_dir / "src")
command = [
    sys.executable, "-m", "aether.experiments.colab_stage6",
    "--allow-download",
    "--model", MODEL_ID,
    "--output-dir", str(artifacts),
]
run = subprocess.run(command, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
(artifacts / "model_run.log").write_text(run.stdout, encoding="utf-8")
print(run.stdout)
print("Exit code:", run.returncode)


In [ ]:
import json

report_path = artifacts / "report.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print("Status:", report.get("status"))
print("Scope note:", report.get("scope_note"))
print("Checks:", json.dumps(report.get("checks"), indent=2))
for name, result in report.get("results", {}).items():
    print("\n===", name, "===")
    print("Request:", result.get("request"))
    print("Thought:", result.get("thought"))
    print("Baseline response:", result.get("baseline_response"))
    print("Conditioned response:", result.get("conditioned_response"))
    print("Cross-conditioned (from", result.get("cross_conditioned_from"), "):", result.get("cross_conditioned_response"))
    print("Checks:", result.get("checks"))


## Если упало

`report.json` пишется на каждом шаге. `traceback.txt` в архиве покажет, где именно (загрузка
модели, генерация мысли, encode_hidden_state, или сам soft-prompt inference через
`generate_with_soft_prompt`).

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/aether-colab-stage6-logs", "zip", root_dir=artifacts)
print(archive)
files.download(archive)
